# Daily Challenge - Global Power Plant Database

**Objectif :** analyser un dataset reel avec **NumPy**, **Pandas**, **Matplotlib** et **Seaborn**.

Ce notebook couvre :
- Import et nettoyage des donnees
- Analyse exploratoire
- Analyse statistique + test d'hypothese
- Analyse temporelle
- Visualisations avancees
- Operations matricielles et interpretation des valeurs/vecteurs propres

In [ ]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# 1) Import des donnees et preparation des fichiers
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / 'globalpowerplantdatabasev130.zip'

zip_url = 'https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%206%20-%20Applications%20for%20Data%20Analysis/W6D2%20-%20Advanced%20Numpy/globalpowerplantdatabasev130.zip'

if not zip_path.exists():
    import requests
    response = requests.get(zip_url, timeout=60)
    response.raise_for_status()
    zip_path.write_bytes(response.content)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(data_dir)

csv_files = list(data_dir.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError('Aucun fichier CSV trouve dans l archive.')

csv_path = csv_files[0]
df = pd.read_csv(csv_path)
print(f'CSV charge : {csv_path}')
print(f'Dimensions initiales : {df.shape}')

In [ ]:
df.head()

In [ ]:
# 2) Identification et traitement des valeurs manquantes
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report[missing_report['missing_count'] > 0].head(15)

In [ ]:
# Conversion numerique avec NumPy/Pandas
numeric_candidates = [
    'capacity_mw', 'latitude', 'longitude', 'commissioning_year',
    'estimated_generation_gwh_2013', 'estimated_generation_gwh_2014',
    'estimated_generation_gwh_2015', 'estimated_generation_gwh_2016',
    'estimated_generation_gwh_2017'
]
numeric_cols = [c for c in numeric_candidates if c in df.columns]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['primary_fuel'] = df['primary_fuel'].fillna('Unknown')
if 'country_long' in df.columns:
    df['country_long'] = df['country_long'].fillna('Unknown')

# Jeu de travail pour analyses quantitatives
required = ['capacity_mw', 'primary_fuel']
df_clean = df.dropna(subset=[c for c in required if c in df.columns]).copy()

print('Dimensions apres nettoyage :', df_clean.shape)
df_clean[numeric_cols].describe().T[['mean', '50%', 'std', 'min', 'max']]

## Analyse Exploratoire
Distribution des centrales par pays et type de carburant principal.

In [ ]:
country_col = 'country_long' if 'country_long' in df_clean.columns else 'country'
top_countries = df_clean[country_col].value_counts().head(15)
top_fuels = df_clean['primary_fuel'].value_counts().head(15)

display(top_countries.to_frame('plant_count'))
display(top_fuels.to_frame('plant_count'))

## Analyse Statistique
Statistiques de capacite (MW) par type de carburant avec NumPy.

In [ ]:
fuel_stats = []
for fuel, group in df_clean.groupby('primary_fuel'):
    arr = group['capacity_mw'].dropna().to_numpy(dtype=float)
    if arr.size < 5:
        continue
    fuel_stats.append({
        'fuel': fuel,
        'count': arr.size,
        'mean_mw': np.mean(arr),
        'median_mw': np.median(arr),
        'std_mw': np.std(arr, ddof=1),
        'q25_mw': np.percentile(arr, 25),
        'q75_mw': np.percentile(arr, 75),
    })

fuel_stats_df = pd.DataFrame(fuel_stats).sort_values('mean_mw', ascending=False)
fuel_stats_df.head(15)

In [ ]:
# 3) Test d'hypothese : permutation test (difference de moyenne)
def permutation_test_mean_diff(x, y, n_perm=5000, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    observed = np.mean(x) - np.mean(y)

    pooled = np.concatenate([x, y])
    n_x = x.size
    diffs = np.empty(n_perm)

    for i in range(n_perm):
        rng.shuffle(pooled)
        diffs[i] = np.mean(pooled[:n_x]) - np.mean(pooled[n_x:])

    p_value = np.mean(np.abs(diffs) >= abs(observed))
    return observed, p_value

top3_fuels = df_clean['primary_fuel'].value_counts().head(3).index.tolist()
pairwise_results = []

for i in range(len(top3_fuels)):
    for j in range(i + 1, len(top3_fuels)):
        f1, f2 = top3_fuels[i], top3_fuels[j]
        x = df_clean.loc[df_clean['primary_fuel'] == f1, 'capacity_mw'].dropna().to_numpy()
        y = df_clean.loc[df_clean['primary_fuel'] == f2, 'capacity_mw'].dropna().to_numpy()
        observed, p_value = permutation_test_mean_diff(x, y)
        pairwise_results.append({
            'fuel_1': f1,
            'fuel_2': f2,
            'mean_diff_mw': observed,
            'p_value': p_value,
            'significant_at_5pct': p_value < 0.05
        })

pd.DataFrame(pairwise_results).sort_values('p_value')

## Analyse Temporelle
Tendance du nombre de centrales et evolution du mix energetique.

In [ ]:
if 'commissioning_year' in df_clean.columns:
    time_df = df_clean[df_clean['commissioning_year'].between(1900, 2030)].copy()
    time_df['commissioning_year'] = time_df['commissioning_year'].astype(int)
    yearly_counts = time_df.groupby('commissioning_year').size()

    # Mix par decennie
    time_df['decade'] = (time_df['commissioning_year'] // 10) * 10
    fuel_mix_decade = pd.crosstab(time_df['decade'], time_df['primary_fuel'])
    fuel_mix_share = fuel_mix_decade.div(fuel_mix_decade.sum(axis=1), axis=0)

    display(yearly_counts.tail(15).to_frame('new_plants'))
    display(fuel_mix_share.tail(10).iloc[:, :10])
else:
    print('La colonne commissioning_year est absente.')

## Visualisations Avancees
Graphiques Matplotlib/Seaborn pour resumer les principaux resultats.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_countries.values, y=top_countries.index, ax=ax[0], palette='viridis')
ax[0].set_title('Top pays par nombre de centrales')
ax[0].set_xlabel('Nombre de centrales')
ax[0].set_ylabel('Pays')

sns.barplot(x=top_fuels.values, y=top_fuels.index, ax=ax[1], palette='mako')
ax[1].set_title('Top carburants (primary_fuel)')
ax[1].set_xlabel('Nombre de centrales')
ax[1].set_ylabel('Carburant')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution de la capacite par fuel (top 10)
top10_fuels = df_clean['primary_fuel'].value_counts().head(10).index
plot_df = df_clean[df_clean['primary_fuel'].isin(top10_fuels)].copy()

plt.figure(figsize=(14, 6))
sns.boxplot(data=plot_df, x='primary_fuel', y='capacity_mw', showfliers=False, palette='crest')
plt.yscale('log')
plt.title('Distribution de capacity_mw par fuel (echelle log)')
plt.xlabel('Fuel')
plt.ylabel('Capacity (MW, log scale)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
if 'commissioning_year' in df_clean.columns:
    plt.figure(figsize=(14, 5))
    yearly_counts.rolling(5, min_periods=1).mean().plot(color='teal', linewidth=2)
    plt.title('Tendance (moyenne mobile 5 ans) du nombre de centrales commissionnees')
    plt.xlabel('Annee')
    plt.ylabel('Nombre de centrales')
    plt.tight_layout()
    plt.show()

    # Heatmap du mix energetique sur les 8 fuels les plus frequents
    top8 = df_clean['primary_fuel'].value_counts().head(8).index
    heat = fuel_mix_share[top8].fillna(0)

    plt.figure(figsize=(12, 6))
    sns.heatmap(heat, cmap='YlGnBu', annot=False)
    plt.title('Evolution du mix energetique par decennie (part relative)')
    plt.xlabel('Fuel')
    plt.ylabel('Decennie')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution geographique (si latitude/longitude disponibles)
geo_cols = {'latitude', 'longitude'}
if geo_cols.issubset(df_clean.columns):
    geo_df = df_clean.dropna(subset=['latitude', 'longitude', 'capacity_mw']).copy()
    geo_df = geo_df.sample(min(3000, len(geo_df)), random_state=42)

    plt.figure(figsize=(13, 6))
    scatter = plt.scatter(
        geo_df['longitude'],
        geo_df['latitude'],
        s=np.sqrt(geo_df['capacity_mw'].clip(lower=1)) * 1.2,
        c=pd.factorize(geo_df['primary_fuel'])[0],
        cmap='tab20',
        alpha=0.55
    )
    plt.title('Distribution geographique des centrales (taille ~ sqrt(capacity_mw))')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.show()

## Operations Matricielles et Valeurs Propres
Construction d'une matrice numerique, covariance et decomposition spectrale.

In [ ]:
matrix_features = [c for c in ['capacity_mw', 'latitude', 'longitude', 'commissioning_year'] if c in df_clean.columns]
mat_df = df_clean[matrix_features].dropna().copy()

X = mat_df.to_numpy(dtype=float)
X_std = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

cov_matrix = np.cov(X_std, rowvar=False)
eig_vals, eig_vecs = np.linalg.eig(cov_matrix)

eig_order = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[eig_order]
eig_vecs = eig_vecs[:, eig_order]
explained_ratio = eig_vals / eig_vals.sum()

cov_df = pd.DataFrame(cov_matrix, index=matrix_features, columns=matrix_features)
eig_df = pd.DataFrame({
    'eigenvalue': eig_vals.real,
    'explained_ratio': explained_ratio.real
})

display(cov_df)
display(eig_df)

In [ ]:
# Projection sur les deux premiers axes principaux
pc_scores = X_std @ eig_vecs[:, :2].real

plt.figure(figsize=(10, 6))
plt.scatter(pc_scores[:, 0], pc_scores[:, 1], alpha=0.2, s=10, color='darkcyan')
plt.title('Projection matricielle sur 2 axes principaux')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

### Pourquoi les valeurs/vecteurs propres sont utiles ici ?
- Les **valeurs propres** mesurent la variance capturee par chaque direction principale.
- Les **vecteurs propres** donnent les combinaisons lineaires d'attributs (capacite, localisation, annee) les plus informatives.
- Cela aide a reduire la dimension et a interpreter les structures dominantes des donnees.

## Integration NumPy + Pandas + Matplotlib
Exemples de filtrage avance et visualisation assistee par NumPy.

In [ ]:
# Exemple 1: filtrage complexe vectorise
cap = df_clean['capacity_mw'].to_numpy()
is_large = cap > np.percentile(cap, 75)
is_mid_age = np.ones_like(cap, dtype=bool)

if 'commissioning_year' in df_clean.columns:
    years = df_clean['commissioning_year'].to_numpy()
    is_mid_age = (years >= 1980) & (years <= 2010)

mask = is_large & is_mid_age
filtered_df = df_clean.loc[mask, ['name', 'country_long', 'primary_fuel', 'capacity_mw'] if 'country_long' in df_clean.columns else ['name', 'country', 'primary_fuel', 'capacity_mw']]

filtered_df.head(10)

In [ ]:
# Exemple 2: histogramme avec bins NumPy personnalises
bins = np.linspace(0, np.percentile(df_clean['capacity_mw'], 99), 30)

plt.figure(figsize=(11, 5))
plt.hist(df_clean['capacity_mw'].clip(upper=bins.max()), bins=bins, color='steelblue', edgecolor='white')
plt.title('Histogramme de la capacite (coupe au 99e percentile)')
plt.xlabel('Capacity MW')
plt.ylabel('Frequence')
plt.tight_layout()
plt.show()

## Conclusion
Ce notebook montre comment combiner NumPy, Pandas et Matplotlib/Seaborn pour une analyse de donnees reelles a grande echelle, du nettoyage a l'inference statistique et a la visualisation.